# 🔍 Silver Layer Data Quality Validation

This notebook performs comprehensive data quality checks on the Silver layer tables to ensure data cleanliness and integrity.

**What this notebook checks:**
- ✅ Record counts and completeness
- ✅ Null value analysis
- ✅ Data type validation
- ✅ Business rule compliance
- ✅ Referential integrity with zone lookup
- ✅ Statistical anomalies
- ✅ Quality flag distribution
- ✅ Temporal consistency

**Author:** Data Engineering Team  
**Last Updated:** February 2026

## 📦 Setup & Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
from datetime import datetime

# Database configuration
DATABASE = "nyc_taxi_lakehouse"

# Silver tables to validate
SILVER_TABLES = {
    "yellow": f"{DATABASE}.silver.yellow_trips",
    "green": f"{DATABASE}.silver.green_trips",
    "fhv": f"{DATABASE}.silver.fhv_trips",
    "fhvhv": f"{DATABASE}.silver.fhvhv_trips"
}

# Zone lookup table
ZONE_LOOKUP_TABLE = f"{DATABASE}.bronze.taxi_zone_lookup"
METADATA_TABLE = f"{DATABASE}.silver.processing_metadata"

# Valid zone IDs (NYC has 265 taxi zones)
VALID_ZONE_RANGE = (1, 265)

print("✅ Configuration loaded!")
print(f"🔍 Will validate {len(SILVER_TABLES)} silver tables")

## 📊 1. High-Level Overview

Let's start with the big picture - how much data do we have and when was it processed?

In [0]:
print("="*80)
print("📈 SILVER LAYER OVERVIEW")
print("="*80)

overview_data = []

for trip_type, table_name in SILVER_TABLES.items():
    try:
        df = spark.table(table_name)
        total_count = df.count()
        
        # Get date range
        date_stats = df.agg(
            F.min("pickup_datetime").alias("earliest"),
            F.max("pickup_datetime").alias("latest")
        ).collect()[0]
        
        # Get partition info
        partitions = df.select("year", "month").distinct().count()
        
        overview_data.append({
            "Trip Type": trip_type.upper(),
            "Total Records": f"{total_count:,}",
            "Partitions": partitions,
            "Earliest Trip": date_stats["earliest"],
            "Latest Trip": date_stats["latest"]
        })
        
    except Exception as e:
        print(f"⚠️  Error reading {trip_type}: {str(e)}")

overview_df = pd.DataFrame(overview_data)
display(overview_df)

In [0]:
# Check processing metadata
print("\n📋 Processing Status by Trip Type and Month:")
metadata_summary = spark.table(METADATA_TABLE) \
    .groupBy("trip_type", "status") \
    .agg(F.count("*").alias("count")) \
    .orderBy("trip_type", "status")

display(metadata_summary)

## 🎯 2. Completeness Check

Let's see how complete our data is - are we missing important fields?

In [0]:
def check_null_percentages(df, trip_type, critical_columns):
    """
    Calculate null percentage for each column.
    Highlight critical columns that shouldn't have nulls.
    """
    total_rows = df.count()
    
    null_stats = []
    for col_name in df.columns:
        null_count = df.filter(F.col(col_name).isNull()).count()
        null_pct = (null_count / total_rows) * 100 if total_rows > 0 else 0
        
        null_stats.append({
            "Column": col_name,
            "Null Count": null_count,
            "Null %": round(null_pct, 2),
            "Critical": "⚠️ YES" if col_name in critical_columns and null_pct > 0 else "No"
        })
    
    # Sort by null percentage descending
    null_df = pd.DataFrame(null_stats).sort_values("Null %", ascending=False)
    
    return null_df

# Define critical columns that should NOT have nulls
CRITICAL_COLUMNS = {
    "yellow": ["pickup_datetime", "dropoff_datetime", "pickup_location_id", "dropoff_location_id"],
    "green": ["pickup_datetime", "dropoff_datetime", "pickup_location_id", "dropoff_location_id"],
    "fhv": ["pickup_datetime", "dropoff_datetime", "dispatching_base_number"],
    "fhvhv": ["pickup_datetime", "dropoff_datetime", "pickup_location_id", "dropoff_location_id"]
}

print("="*80)
print("📊 NULL VALUE ANALYSIS")
print("="*80)

In [0]:
# Check Yellow trips
print("\n🚕 YELLOW TRIPS - Null Analysis")
yellow_df = spark.table(SILVER_TABLES["yellow"])
yellow_nulls = check_null_percentages(yellow_df, "yellow", CRITICAL_COLUMNS["yellow"])
display(yellow_nulls.head(20))  # Show top 20 columns with most nulls

In [0]:
# Check Green trips
print("\n🚕 GREEN TRIPS - Null Analysis")
green_df = spark.table(SILVER_TABLES["green"])
green_nulls = check_null_percentages(green_df, "green", CRITICAL_COLUMNS["green"])
display(green_nulls.head(20))

In [0]:
# Check FHV trips
print("\n🚗 FHV TRIPS - Null Analysis")
fhv_df = spark.table(SILVER_TABLES["fhv"])
fhv_nulls = check_null_percentages(fhv_df, "fhv", CRITICAL_COLUMNS["fhv"])
display(fhv_nulls.head(20))

In [0]:
# Check FHVHV trips
print("\n🚙 FHVHV TRIPS - Null Analysis")
fhvhv_df = spark.table(SILVER_TABLES["fhvhv"])
fhvhv_nulls = check_null_percentages(fhvhv_df, "fhvhv", CRITICAL_COLUMNS["fhvhv"])
display(fhvhv_nulls.head(20))

## ✅ 3. Business Rule Validation

Let's verify that our data follows expected business rules and constraints.

In [0]:
print("="*80)
print("✅ BUSINESS RULE VALIDATION")
print("="*80)

def validate_business_rules(df, trip_type):
    """
    Check various business rules and return violation counts.
    """
    total = df.count()
    rules = []
    
    # Rule 1: Trip duration should be positive
    negative_duration = df.filter(F.col("trip_duration_seconds") < 0).count()
    rules.append({
        "Rule": "Trip duration >= 0",
        "Violations": negative_duration,
        "% of Total": round((negative_duration/total)*100, 2) if total > 0 else 0,
        "Status": "✅ PASS" if negative_duration == 0 else "❌ FAIL"
    })
    
    # Rule 2: Pickup should be before dropoff
    invalid_time_order = df.filter(F.col("pickup_datetime") >= F.col("dropoff_datetime")).count()
    rules.append({
        "Rule": "Pickup before dropoff",
        "Violations": invalid_time_order,
        "% of Total": round((invalid_time_order/total)*100, 2) if total > 0 else 0,
        "Status": "✅ PASS" if invalid_time_order == 0 else "❌ FAIL"
    })
    
    # Rule 3: Location IDs in valid range (if applicable)
    if trip_type in ["yellow", "green", "fhvhv"]:
        invalid_pickup_loc = df.filter(
            (F.col("pickup_location_id") < VALID_ZONE_RANGE[0]) | 
            (F.col("pickup_location_id") > VALID_ZONE_RANGE[1])
        ).count()
        
        invalid_dropoff_loc = df.filter(
            (F.col("dropoff_location_id") < VALID_ZONE_RANGE[0]) | 
            (F.col("dropoff_location_id") > VALID_ZONE_RANGE[1])
        ).count()
        
        rules.append({
            "Rule": "Valid pickup location (1-265)",
            "Violations": invalid_pickup_loc,
            "% of Total": round((invalid_pickup_loc/total)*100, 2) if total > 0 else 0,
            "Status": "✅ PASS" if invalid_pickup_loc == 0 else "❌ FAIL"
        })
        
        rules.append({
            "Rule": "Valid dropoff location (1-265)",
            "Violations": invalid_dropoff_loc,
            "% of Total": round((invalid_dropoff_loc/total)*100, 2) if total > 0 else 0,
            "Status": "✅ PASS" if invalid_dropoff_loc == 0 else "❌ FAIL"
        })
    
    # Rule 4: Financial validation (for yellow and green)
    if trip_type in ["yellow", "green"]:
        negative_fare = df.filter(F.col("total_fare") < 0).count()
        rules.append({
            "Rule": "Total fare >= 0",
            "Violations": negative_fare,
            "% of Total": round((negative_fare/total)*100, 2) if total > 0 else 0,
            "Status": "✅ PASS" if negative_fare == 0 else "❌ FAIL"
        })
        
        negative_distance = df.filter(F.col("trip_distance_miles") < 0).count()
        rules.append({
            "Rule": "Distance >= 0",
            "Violations": negative_distance,
            "% of Total": round((negative_distance/total)*100, 2) if total > 0 else 0,
            "Status": "✅ PASS" if negative_distance == 0 else "❌ FAIL"
        })
        
        unreasonable_distance = df.filter(F.col("trip_distance_miles") > 500).count()
        rules.append({
            "Rule": "Distance <= 500 miles",
            "Violations": unreasonable_distance,
            "% of Total": round((unreasonable_distance/total)*100, 2) if total > 0 else 0,
            "Status": "✅ PASS" if unreasonable_distance == 0 else "❌ FAIL"
        })
    
    # Rule 5: Future dates check
    future_dates = df.filter(F.col("pickup_datetime") > F.current_timestamp()).count()
    rules.append({
        "Rule": "No future pickup dates",
        "Violations": future_dates,
        "% of Total": round((future_dates/total)*100, 2) if total > 0 else 0,
        "Status": "✅ PASS" if future_dates == 0 else "❌ FAIL"
    })
    
    return pd.DataFrame(rules)

In [0]:
# Validate Yellow trips
print("\n🚕 YELLOW TRIPS - Business Rules")
yellow_rules = validate_business_rules(yellow_df, "yellow")
display(yellow_rules)

In [0]:
# Validate Green trips
print("\n🚕 GREEN TRIPS - Business Rules")
green_rules = validate_business_rules(green_df, "green")
display(green_rules)

In [0]:
# Validate FHV trips
print("\n🚗 FHV TRIPS - Business Rules")
fhv_rules = validate_business_rules(fhv_df, "fhv")
display(fhv_rules)

In [0]:
# Validate FHVHV trips
print("\n🚙 FHVHV TRIPS - Business Rules")
fhvhv_rules = validate_business_rules(fhvhv_df, "fhvhv")
display(fhvhv_rules)

## 🔗 4. Referential Integrity

Let's verify that our location IDs properly reference the zone lookup table.

In [0]:
print("="*80)
print("🔗 REFERENTIAL INTEGRITY CHECK")
print("="*80)

# Load zone lookup
zone_lookup = spark.table(ZONE_LOOKUP_TABLE)
valid_zone_ids = [row.LocationID for row in zone_lookup.select("LocationID").collect()]
print(f"\n📍 Total valid zone IDs: {len(valid_zone_ids)}")

def check_zone_integrity(df, trip_type):
    """
    Check if pickup and dropoff zones exist in the zone lookup table.
    """
    total = df.count()
    
    # Check for orphaned pickup locations
    orphaned_pickup = df.filter(
        F.col("pickup_location_id").isNotNull() & 
        F.col("pickup_borough").isNull()
    ).count()
    
    # Check for orphaned dropoff locations
    orphaned_dropoff = df.filter(
        F.col("dropoff_location_id").isNotNull() & 
        F.col("dropoff_borough").isNull()
    ).count()
    
    results = {
        "Trip Type": trip_type.upper(),
        "Total Records": f"{total:,}",
        "Orphaned Pickup Zones": orphaned_pickup,
        "Orphaned Dropoff Zones": orphaned_dropoff,
        "Pickup Integrity": "✅ PASS" if orphaned_pickup == 0 else "⚠️ ISSUES",
        "Dropoff Integrity": "✅ PASS" if orphaned_dropoff == 0 else "⚠️ ISSUES"
    }
    
    return results

integrity_results = []

for trip_type in ["yellow", "green", "fhvhv"]:
    df = spark.table(SILVER_TABLES[trip_type])
    result = check_zone_integrity(df, trip_type)
    integrity_results.append(result)

integrity_df = pd.DataFrame(integrity_results)
display(integrity_df)

## 📈 5. Statistical Analysis

Let's look at the statistical distribution of key metrics to spot anomalies.

In [0]:
print("="*80)
print("📈 STATISTICAL ANALYSIS - YELLOW TRIPS")
print("="*80)

# Yellow trip statistics
yellow_stats = yellow_df.select(
    "trip_distance_miles",
    "trip_duration_minutes",
    "total_fare",
    "passenger_count",
    "average_speed_mph",
    "tip_percentage"
).describe()

display(yellow_stats)

In [0]:
print("="*80)
print("📈 STATISTICAL ANALYSIS - GREEN TRIPS")
print("="*80)

# Green trip statistics
green_stats = green_df.select(
    "trip_distance_miles",
    "trip_duration_minutes",
    "total_fare",
    "passenger_count",
    "average_speed_mph",
    "tip_percentage"
).describe()

display(green_stats)

In [0]:
# Check for extreme outliers in Yellow trips
print("\n🔍 Extreme Values in Yellow Trips:")

outliers = yellow_df.filter(
    (F.col("trip_distance_miles") > 100) |
    (F.col("trip_duration_minutes") > 300) |
    (F.col("total_fare") > 500) |
    (F.col("average_speed_mph") > 100)
).count()

print(f"Records with extreme values: {outliers:,}")
print(f"Percentage of total: {(outliers/yellow_df.count())*100:.2f}%")

## 🚩 6. Data Quality Flags Analysis

Let's examine the quality flags we created during transformation.

In [0]:
print("="*80)
print("🚩 DATA QUALITY FLAGS DISTRIBUTION")
print("="*80)

def analyze_quality_flags(df, trip_type):
    """
    Analyze the distribution of quality flags.
    """
    total = df.count()
    suspicious = df.filter(F.col("is_suspicious") == True).count()
    
    print(f"\n📊 {trip_type.upper()} Trips:")
    print(f"   Total records: {total:,}")
    print(f"   Suspicious records: {suspicious:,} ({(suspicious/total)*100:.2f}%)")
    print(f"   Clean records: {total - suspicious:,} ({((total-suspicious)/total)*100:.2f}%)")
    
    # Get flag breakdown
    flag_dist = df.filter(F.col("is_suspicious") == True) \
        .select(F.explode("data_quality_flags").alias("flag")) \
        .groupBy("flag") \
        .count() \
        .withColumn("percentage", (F.col("count") / total) * 100) \
        .orderBy(F.col("count").desc())
    
    return flag_dist

# Analyze Yellow trips
yellow_flags = analyze_quality_flags(yellow_df, "yellow")
display(yellow_flags)

In [0]:
# Analyze Green trips
green_flags = analyze_quality_flags(green_df, "green")
display(green_flags)

In [0]:
# Analyze FHVHV trips
fhvhv_flags = analyze_quality_flags(fhvhv_df, "fhvhv")
display(fhvhv_flags)

## ⏰ 7. Temporal Consistency

Let's check if our temporal fields are consistent and make sense.

In [0]:
print("="*80)
print("⏰ TEMPORAL CONSISTENCY CHECK")
print("="*80)

def check_temporal_consistency(df, trip_type):
    """
    Verify temporal fields are derived correctly.
    """
    # Sample some records and verify year/month match
    sample = df.select(
        "pickup_datetime",
        "year",
        "month",
        "pickup_day",
        "pickup_hour",
        "pickup_day_name",
        "pickup_is_weekend"
    ).limit(10)
    
    # Verify year matches
    year_mismatch = df.filter(
        F.year("pickup_datetime") != F.col("year")
    ).count()
    
    # Verify month matches
    month_mismatch = df.filter(
        F.month("pickup_datetime") != F.col("month")
    ).count()
    
    # Verify weekend flag
    weekend_mismatch = df.filter(
        (F.col("pickup_is_weekend") == True) & ~(F.col("pickup_day_of_week").isin([1, 7]))
    ).count()
    
    results = {
        "Check": ["Year matches pickup_datetime", "Month matches pickup_datetime", "Weekend flag correct"],
        "Mismatches": [year_mismatch, month_mismatch, weekend_mismatch],
        "Status": [
            "✅ PASS" if year_mismatch == 0 else "❌ FAIL",
            "✅ PASS" if month_mismatch == 0 else "❌ FAIL",
            "✅ PASS" if weekend_mismatch == 0 else "❌ FAIL"
        ]
    }
    
    return pd.DataFrame(results)

print("\n🚕 YELLOW TRIPS - Temporal Consistency")
yellow_temporal = check_temporal_consistency(yellow_df, "yellow")
display(yellow_temporal)

In [0]:
# Distribution by time of day
print("\n📅 Trip Distribution by Time of Day (Yellow):")
time_dist = yellow_df.groupBy("pickup_time_of_day") \
    .count() \
    .withColumn("percentage", (F.col("count") / yellow_df.count()) * 100) \
    .orderBy(F.col("count").desc())

display(time_dist)

In [0]:
# Distribution by day of week
print("\n📅 Trip Distribution by Day of Week (Yellow):")
day_dist = yellow_df.groupBy("pickup_day_name") \
    .count() \
    .withColumn("percentage", (F.col("count") / yellow_df.count()) * 100) \
    .orderBy(F.col("count").desc())

display(day_dist)

## 🗺️ 8. Geographic Distribution

Let's look at the most popular pickup and dropoff zones.

In [0]:
print("="*80)
print("🗺️ GEOGRAPHIC DISTRIBUTION")
print("="*80)

# Top pickup boroughs for Yellow
print("\n📍 Top Pickup Boroughs - Yellow Trips:")
top_pickup_boroughs = yellow_df.groupBy("pickup_borough") \
    .count() \
    .withColumn("percentage", (F.col("count") / yellow_df.count()) * 100) \
    .orderBy(F.col("count").desc()) \
    .limit(10)

display(top_pickup_boroughs)

In [0]:
# Top pickup zones for Yellow
print("\n📍 Top 15 Pickup Zones - Yellow Trips:")
top_pickup_zones = yellow_df.groupBy("pickup_zone", "pickup_borough") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(15)

display(top_pickup_zones)

In [0]:
# Airport trips analysis
print("\n✈️ Airport-Related Trips:")

airport_keywords = ["JFK", "LaGuardia", "Newark", "Airport"]

airport_trips = yellow_df.filter(
    F.col("pickup_zone").rlike("(?i)" + "|".join(airport_keywords)) |
    F.col("dropoff_zone").rlike("(?i)" + "|".join(airport_keywords))
).count()

total_trips = yellow_df.count()

print(f"Airport-related trips: {airport_trips:,}")
print(f"Percentage of total: {(airport_trips/total_trips)*100:.2f}%")

## 💰 9. Financial Data Validation

Let's examine payment methods, fares, and tips to ensure financial data makes sense.

In [0]:
print("="*80)
print("💰 FINANCIAL DATA VALIDATION")
print("="*80)

# Payment method distribution for Yellow
print("\n💳 Payment Method Distribution - Yellow Trips:")
payment_dist = yellow_df.groupBy("payment_method") \
    .count() \
    .withColumn("percentage", (F.col("count") / yellow_df.count()) * 100) \
    .orderBy(F.col("count").desc())

display(payment_dist)

In [0]:
# Tip analysis for credit card payments
print("\n💵 Tip Analysis (Credit Card Payments Only):")
tip_stats = yellow_df.filter(F.col("payment_method") == "Credit Card") \
    .select("tip_amount", "tip_percentage") \
    .describe()

display(tip_stats)

In [0]:
# Zero fare trips
print("\n🆓 Zero Fare Trips:")
zero_fare = yellow_df.filter(F.col("total_fare") == 0).count()
print(f"Trips with zero fare: {zero_fare:,}")
print(f"Percentage: {(zero_fare/yellow_df.count())*100:.2f}%")

# Show payment methods for zero fare trips
zero_fare_payments = yellow_df.filter(F.col("total_fare") == 0) \
    .groupBy("payment_method") \
    .count() \
    .orderBy(F.col("count").desc())

display(zero_fare_payments)

## 🎯 10. Category Distribution Analysis

Let's analyze our derived categories to ensure they're being populated correctly.

In [0]:
print("="*80)
print("🎯 CATEGORY DISTRIBUTION ANALYSIS")
print("="*80)

# Distance categories
print("\n📏 Distance Category Distribution - Yellow Trips:")
distance_cat_dist = yellow_df.groupBy("distance_category") \
    .count() \
    .withColumn("percentage", (F.col("count") / yellow_df.count()) * 100) \
    .orderBy(F.col("count").desc())

display(distance_cat_dist)

In [0]:
# Passenger categories
print("\n👥 Passenger Category Distribution - Yellow Trips:")
passenger_cat_dist = yellow_df.groupBy("passenger_category") \
    .count() \
    .withColumn("percentage", (F.col("count") / yellow_df.count()) * 100) \
    .orderBy(F.col("count").desc())

display(passenger_cat_dist)

In [0]:
# Rate code distribution
print("\n💵 Rate Type Distribution - Yellow Trips:")
rate_type_dist = yellow_df.groupBy("rate_type") \
    .count() \
    .withColumn("percentage", (F.col("count") / yellow_df.count()) * 100) \
    .orderBy(F.col("count").desc())

display(rate_type_dist)

## 🚙 11. FHVHV Specific Analysis

Let's analyze ride-sharing specific fields (Uber & Lyft).

In [0]:
print("="*80)
print("🚙 FHVHV (UBER & LYFT) ANALYSIS")
print("="*80)

# Company distribution
print("\n🏢 Company Distribution:")
company_dist = fhvhv_df.groupBy("company_name") \
    .count() \
    .withColumn("percentage", (F.col("count") / fhvhv_df.count()) * 100) \
    .orderBy(F.col("count").desc())

display(company_dist)

## 📋 12. Final Summary Report

Let's create a comprehensive summary of our data quality validation.

In [0]:
print("="*80)
print("📋 FINAL DATA QUALITY SUMMARY REPORT")
print("="*80)
print(f"\nReport Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Database: {DATABASE}")
print("\n" + "="*80)

summary_report = []

for trip_type, table_name in SILVER_TABLES.items():
    df = spark.table(table_name)
    total = df.count()
    suspicious = df.filter(F.col("is_suspicious") == True).count()
    
    summary_report.append({
        "Trip Type": trip_type.upper(),
        "Total Records": f"{total:,}",
        "Clean Records": f"{total - suspicious:,}",
        "Flagged Records": f"{suspicious:,}",
        "Quality Score": f"{((total - suspicious)/total)*100:.2f}%" if total > 0 else "N/A",
        "Status": "✅ GOOD" if ((total - suspicious)/total)*100 > 95 else "⚠️ REVIEW"
    })

summary_df = pd.DataFrame(summary_report)
display(summary_df)

print("\n" + "="*80)
print("🎉 Data Quality Validation Complete!")
print("="*80)
print("\n📝 Recommendations:")
print("   1. Review records with quality flags for patterns")
print("   2. Investigate any business rule violations")
print("   3. Monitor null percentages for critical fields")
print("   4. Validate zone lookup coverage is complete")
print("   5. Check temporal consistency across partitions")
print("\n💡 For detailed investigation, refer to individual sections above.")

---

## 🎯 Next Steps

Based on the validation results:

**If quality score > 95%:**
- ✅ Data is clean and ready for Gold layer transformation
- ✅ Proceed with analytics and reporting

**If quality score < 95%:**
- ⚠️ Review flagged records in detail
- ⚠️ Investigate patterns in data quality issues
- ⚠️ Consider re-running Bronze → Silver transformation with adjusted rules
- ⚠️ Document known data quality issues for analysts

**Regular monitoring:**
- 📊 Run this notebook after each Silver layer update
- 📊 Track quality scores over time
- 📊 Alert on quality score drops

---